# 03 — Disruption and Replanning

**HYPAQ Ports · Initial educational reference · Apache-2.0**

Start with the berth schedule from Exercise 01 and delay V1 by one slot.
Compare the original plan with free reoptimization and a stability-aware plan.

Replanning happens offline before service starts, so every assignment can change.
There are no frozen commitments. Stability counts vessels whose `(berth, slot)`
pair changes, not the magnitude of the time shift.

**Implemented:** an explicit arrival-delay scenario, classical exact solutions,
and independent validation. **Planned:** additional disruptions, visualizations,
quantum experiments and AI-assisted scenario input.

For Colab, first follow `notebooks/README.md`. Run cells in order.

In [1]:
from pathlib import Path
import sys
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/hypaq_ports/reference.py").is_file()
             and (p / "data/synthetic").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository; see notebooks/README.md for Colab setup.")
sys.path.insert(0, str(ROOT / "src"))
from hypaq_ports.reference import (
    load_json, solve_berth_exact, berth_metrics, validate_berth_schedule,
    build_berth_qubo, qubo_energy, decode_berth_bits, qubo_to_ising,
    solve_crane_exact, evaluate_crane_schedule, apply_arrival_delay,
)
print("Repository:", ROOT.name)


Repository: hypaq-ports-starter


## 1. Establish the original plan and apply a structured event

Synthetic input is copied before updating an arrival. A language model is not used in this starter.

In [2]:
instance = load_json(ROOT / "data/synthetic/berth_tiny_v1.json")
scenario = load_json(ROOT / "data/synthetic/disruption_tiny_v1.json")
original = solve_berth_exact(instance)
event = scenario["event"]
assert event["type"] == "arrival_delay"
updated = apply_arrival_delay(instance, event["vessel"], event["delay_slots"])
original_check = berth_metrics(updated, original["schedule"])
print(json.dumps({"original_schedule": original["schedule"],
                  "original_plan_under_disruption": original_check}, indent=2))
assert not original_check["feasible"]
assert instance["vessels"][0]["arrival_slot"] == 0


{
  "original_schedule": [
    {
      "vessel": "V1",
      "berth": "B1",
      "slot": 0
    },
    {
      "vessel": "V2",
      "berth": "B1",
      "slot": 1
    }
  ],
  "original_plan_under_disruption": {
    "feasible": false,
    "violations": [
      "Service before arrival: V1"
    ],
    "total_waiting_slots": null,
    "changed_vessels": null
  }
}


## 2. Compare free and stability-aware replanning

Free reoptimization minimizes updated waiting time. Stability-aware replanning
minimizes `waiting_slots + weight * changed_vessels`. The weight is expressed in
waiting slots per changed vessel; it is not a measured monetary cost.

Multiple waiting-time optima exist. Input-order tie-breaking makes free
reoptimization choose two changes in this example. A stability penalty selects
one change without increasing waiting time. This illustrates choosing among
equivalent optima; it does not prove a general operational benefit.

In [3]:
free = solve_berth_exact(updated, original=original["schedule"])
stable = solve_berth_exact(updated, original=original["schedule"],
                           change_weight=scenario["change_weight_waiting_slots_per_vessel"])
print(json.dumps({"free": free, "stable": stable}, indent=2))
assert free["metrics"]["feasible"] and stable["metrics"]["feasible"]
assert free["metrics"]["total_waiting_slots"] == stable["metrics"]["total_waiting_slots"] == 0
assert stable["metrics"]["changed_vessels"] < free["metrics"]["changed_vessels"]


{
  "free": {
    "status": "OPTIMAL",
    "method": "exhaustive_enumeration",
    "schedule": [
      {
        "vessel": "V1",
        "berth": "B1",
        "slot": 1
      },
      {
        "vessel": "V2",
        "berth": "B2",
        "slot": 1
      }
    ],
    "metrics": {
      "feasible": true,
      "violations": [],
      "total_waiting_slots": 0,
      "changed_vessels": 2
    },
    "score": 0,
    "change_weight": 0,
    "assignments_examined": 4
  },
  "stable": {
    "status": "OPTIMAL",
    "method": "exhaustive_enumeration",
    "schedule": [
      {
        "vessel": "V1",
        "berth": "B2",
        "slot": 1
      },
      {
        "vessel": "V2",
        "berth": "B1",
        "slot": 1
      }
    ],
    "metrics": {
      "feasible": true,
      "violations": [],
      "total_waiting_slots": 0,
      "changed_vessels": 1
    },
    "score": 2,
    "change_weight": 2,
    "assignments_examined": 4
  }
}


## Explore next

1. Increase the delay beyond the horizon and explain the infeasible status.
2. Add slots or a berth outage and rerun all three evaluations.
3. Construct a larger case where reducing changes increases waiting time.
4. Add a commitment rule that freezes already started services.
5. Later, let an AI assistant propose a validated JSON scenario for user review.

Always report the operational waiting cost separately from the stability penalty.
Do not compare penalized scores as if they were the same objective across different
weights. A plan found infeasible under new data has no valid waiting-time score.